# Precision Agriculture Analysis Pipeline

This notebook acts as an end-to-end execution pipeline utilizing the `Precision Agriculture Spatial Toolkit`.
It sequentially processes raw shapefiles into a regularized spatial grid, models yield outcomes using spatial cross-validation, and interprets the results using SHAP and PDPs.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.append('../scripts')


# 1. Data IO
from data_imports import files_to_dict, cast_to_crs, load_schema, rename_columns_fuzzy
# 2. Preprocessing
from preprocessing import trial_boundaries, remove_boundary_data, filter_parameter, remove_local_outliers, remove_spatial_outliers, text_encoder
# 3. Sensor Delay Correction
from delay import run_robust_pcdi, manual_lag_correction, prepare_shifted_data
# 4. Topography
from terrain import terrain
# 5. Spatial Aggregation
from aggregation import calculate_dominant_heading, create_spatial_grid, aggregate_points_to_grid, assign_polygon_to_grid, align_to_grid, add_spatial_lag, kmeans_clustering, impute_spatial_mean
# 6. Spatial Modeling
from modelling import determine_optimal_block_size, spatial_checkerboard_split, plot_checkerboard_split, rf_predict_spatial, xgb_predict_spatial, compare_spatial_models, tune_spatial_optuna, train_final_model
# 7. Interpretability
from interpretation import run_full_diagnostics, shap_values, plot_three_pdp_ice, two_dim_pdp_plots

%matplotlib inline

## 1. Data Imports & Standardization
Load raw shapefiles from a target directory, align their columns to a standard schema, and assign a uniform Coordinate Reference System (CRS).

In [ ]:
# NOTE: Replace 'data_folder' with the actual path to your shapefiles directory
data_dict, file_names = files_to_dict('../data_folder')

# Load standardization schema for column mapping
schema = load_schema('../scripts/schema_mapping.json')

# Standardize column names using rapidfuzz string matching
for name, df in data_dict.items():
    data_dict[name] = rename_columns_fuzzy(df, schema)

# Cast everything to a metric CRS (e.g., EPSG:32734 for UTM Zone 34S)
data_dict = cast_to_crs(data_dict, crs='EPSG:32734')

# Extract target dataframes for pipeline demonstration 
# (Replace the dictionary keys with your actual file names)
yield_df = data_dict.get('yield_data') 
boundary_df = data_dict.get('field_boundary') 
seed_df = data_dict.get('seeding_data')

## 2. Preprocessing & Cleaning
Filter out non-trial areas, headlands, statistical outliers, and spatial anomalies to ensure the model trains on high-quality agronomic data.

In [ ]:
# Extract main trial area and remove headland/boundary noise (assuming boundary_df exists)
if boundary_df is not None:
    trial_poly = trial_boundaries(boundary_df, plots=True)
    yield_clean, yield_removed = remove_boundary_data(yield_df, trial_poly, buffer_zone=5, plots=True, colname='yield_mass_dry')
else:
    yield_clean = yield_df.copy()

# Filter statistical outliers (e.g., keeping yield values strictly between the 5th and 95th percentile)
yield_clean = filter_parameter(yield_clean, seed_colname='yield_mass_dry', parameter_colname='yield_mass_dry', method='percentile', lower_threshold=0.05, upper_threshold=0.95, plots=True)

# Remove spatial anomalies (spikes or drops compared to immediate neighbors)
yield_clean, local_outliers = remove_local_outliers(yield_clean, value_col='yield_mass_dry', crit='dist', s=15, sd_threshold=2.0, plot_removed=True, plot_example=True)

# Iteratively remove isolated GPS points that lack sufficient spatial density (less than 4 neighbors within 15m)
yield_clean = remove_spatial_outliers(yield_clean, dist=15, n=4, plots=True, value_col='yield_mass_dry')


## 3. Sensor Delay Correction
Correct the spatial offset caused by the time it takes grain to travel from the harvester header to the mass flow sensor.

In [ ]:
# Calculate optimal sensor delay using the Phase Correlation (PCDI) method over multiple iterations
best_delay, rsc_results, delay_std = run_robust_pcdi(
    df=yield_clean,
    header_width=10.0,
    delay_min=-30,
    delay_max=30,
    time_type='index', # Change to 'timestamp' if a valid time_col is available
    yield_col='yield_mass_dry',
    polling_rate=1.0,
    plot_final=True,
    plot_map=True
)

# Apply the calculated temporal shift to align yield mass with true GPS coordinates
yield_shifted = prepare_shifted_data(yield_clean, delay=best_delay, time_type='index', yield_col='yield_mass_dry')


## 4. Terrain Extraction
Derive slope and aspect attributes mathematically from the localized Z-coordinates of the data.

In [ ]:
# Generate terrain derivatives (slope and aspect) from Z-coordinates
# Ensure your yield dataframe has a valid elevation/Z column. If not, map this to an external DEM.
if 'elevation' in yield_shifted.columns:
    terrain_df = terrain(
        x=yield_shifted.geometry.x,
        y=yield_shifted.geometry.y,
        z=yield_shifted['elevation'], 
        crit='dist',
        s=20 # 20m search radius for fitting the 3D plane
    )

    # Join terrain data back to the shifted yield points
    yield_shifted['slope_deg'] = terrain_df['slope_deg']
    yield_shifted['aspect_deg'] = terrain_df['aspect_deg']
else:
    print("Warning: 'elevation' column not found. Skipping terrain processing.")


## 5. Gridding & Aggregation
Standardize variable-density point clouds and management polygons into a uniform ML-ready master grid.

In [ ]:
# Calculate the primary direction the harvester was moving
heading = calculate_dominant_heading(yield_shifted, plots=True)

# Create a spatial grid aligned to the harvester heading
grid, cell_width = create_spatial_grid(
    input_df=yield_shifted,
    cell_size=10.0, # 10m x 10m grid cells
    shape='square',
    angle=heading,
    plots=True,
    value_col='yield_mass_dry',
    boundary_df=boundary_df
)

# Aggregate the intensive point data into the grid cells
master_grid = aggregate_points_to_grid(yield_shifted, grid, value_col='yield_mass_dry', is_rate=True)

if 'elevation' in yield_shifted.columns:
    master_grid = aggregate_points_to_grid(yield_shifted, master_grid, value_col='elevation', is_rate=True)
    master_grid = aggregate_points_to_grid(yield_shifted, master_grid, value_col='slope_deg', is_rate=True)
    master_grid = aggregate_points_to_grid(yield_shifted, master_grid, value_col='aspect_deg', is_rate=True)

# Assign management polygons (e.g., variable rate seeding zones) via area-weighting
if seed_df is not None:
    master_grid = assign_polygon_to_grid(seed_df, master_grid, value_col='applied_rate')

# Impute missing grid values using spatial neighbors to ensure continuous data
master_grid['yield_mass_dry'] = impute_spatial_mean(master_grid, 'yield_mass_dry', method='queen')

if 'elevation' in master_grid.columns:
    master_grid['elevation'] = impute_spatial_mean(master_grid, 'elevation', method='queen')
    master_grid['slope_deg'] = impute_spatial_mean(master_grid, 'slope_deg', method='queen')

# Add a spatial lag column to capture neighborhood context for ML
if 'applied_rate' in master_grid.columns:
    master_grid = add_spatial_lag(master_grid, 'applied_rate', radius=30.0)

# Segment the field into management zones based on key variables
if 'elevation' in master_grid.columns:
    master_grid = kmeans_clustering(master_grid, cols=['yield_mass_dry', 'elevation'], name='zone', auto_k=True, plots=True)


## 6. Spatial Modeling
Train predictive machine learning models strictly using Spatial Block Cross-Validation to account for spatial autocorrelation and avoid over-estimating model performance.

In [ ]:
# Define predictors and target based on available aggregated columns
predictors = []
if 'elevation' in master_grid.columns: predictors.extend(['elevation', 'slope_deg', 'aspect_deg'])
if 'applied_rate' in master_grid.columns: predictors.extend(['applied_rate', 'applied_rate_spatial_lag'])

target = 'yield_mass_dry'

# Drop missing values just for modelling
model_grid = master_grid.dropna(subset=predictors + [target]).copy()

if len(predictors) > 0:
    # Determine the mathematically optimal block size for spatial cross-validation
    opt_block_size = determine_optimal_block_size(model_grid, target, harvester_width=10)

    # Split the field into a checkerboard pattern for spatial hold-out validation
    X_train, X_test, y_train, y_test, train_df, test_df = spatial_checkerboard_split(
        model_grid, predictors, target, block_size=opt_block_size, test_size=0.2
    )

    plot_checkerboard_split(train_df, test_df)

    # Train a baseline Random Forest model with spatial cross-validation
    rf_model, rf_results = rf_predict_spatial(model_grid, predictors, target, block_size=opt_block_size)

    # Tune hyperparameters using Bayesian Optimization (Optuna) for an XGBoost Regressor
    best_xgb_model, study = tune_spatial_optuna(model_grid, predictors, target, model_type='xgb', n_trials=20, block_size=opt_block_size)

    # Train the final production model on 100% of the data using the best parameters found
    final_model = train_final_model(model_grid, predictors, target, model_type='xgb', model_params=study.best_params)
else:
    print("No valid predictors available to train the model.")


## 7. Model Interpretation
Unpack the trained model to understand what agronomic factors drove the yield predictions, utilizing global and local SHAP explanations alongside Partial Dependence Plots.

In [ ]:
if len(predictors) > 0:
    # Provide fallback column names for generic diagnostics
    fert_proxy = 'applied_rate' if 'applied_rate' in predictors else predictors[0]
    seed_proxy = 'applied_rate_spatial_lag' if 'applied_rate_spatial_lag' in predictors else predictors[-1]

    # Run a full suite of diagnostic plots on the holdout set
    run_full_diagnostics(
        final_model,
        X_train, y_train,
        X_test, y_test,
        feature_names=predictors,
        seed_col=seed_proxy,
        fert_col=fert_proxy
    )

    # Generate SHAP explanations to understand global and local feature impacts
    # (pixel_idx_loc isolates the SHAP waterfall plot to a specific coordinate grid cell)
    shap_values(final_model, model_grid, pred_cols=predictors, value_col=target, pixel_idx_loc=10)

    # Visualize 1D Partial Dependence Plots (PDP) and ICE curves for the top features
    if len(predictors) >= 3:
        target_feats = predictors[:3]
        plot_three_pdp_ice(final_model, X_train, target_features=target_feats)

        # Visualize 2D interaction effects on predicted yield
        feat_combos = np.array([
            [target_feats[0], target_feats[1]],
            [target_feats[0], target_feats[2]]
        ])
        two_dim_pdp_plots(final_model, X_train, feat_combos)
    else:
        print("Not enough predictors to render 3-way PDP charts.")
